# Testing Runtime of DBSCAN on Various Sample Sizes in Kaggle
Bert base embeddings from the last layer  
Sample Sizes tested: 500, 5000, 10000, 20000, 40000, 60000, 80000, 100000  
DBSCAN parameters set to epsilon = 0.2 and MinPts = 380

In [ ]:
# Packages
import json
from pathlib import Path
import math
import pandas as pd
import gc 
from sklearn.cluster import DBSCAN
import numpy
import collections
import time
from datetime import datetime

In [24]:
# Define path to the embeddings data
embeddings_path = Path("/kaggle/input/datasets/lianestrauch/bert-base")

## Function to run the DBSCAN algorithm

In [25]:
def get_clusters(sample_size, epsilon, minPts):
    """ 
    Cluster a sample of embeddings using DBSCAN with cosine distance. 
    
    The function loads enough embedding chunks to obtain approximately ``sample_size`` samples, 
    truncates the resulting dataset to exactly ``sample_size`` samples when 
    necessary, and applies DBSCAN clustering using cosine distance. 
    The frequency of each resulting cluster label is returned. 
    
    Parameters 
    ---------- 
    sample_size : int 
        Number of samples to use for clustering. 
        Embeddings are loaded from parquet files in chunks of approximately 4,000 samples.
    
    epsilon : float 
        Maximum distance between two samples for them to be considered neighbors by DBSCAN. 
    
    minPts : int 
        Minimum number of samples in an epsilon neighborhood required for a point 
        to be considered a core point by DBSCAN. 
        
    Returns 
    ------- 
    collections.Counter 
        Counter containing the number of samples assigned to each cluster label. 
        The label ``-1`` represents noise points that were not assigned to a cluster. 
    
    Notes 
    ----- 
    Embeddings are loaded from the ``layer_last`` column of parquet files 
    matching ``chunk_*.parquet`` in ``embeddings_path``. 
    
    DBSCAN is performed using cosine distance. 
    
    """
    
    ########### SAMPLE ########################
    nr_of_files = math.ceil(sample_size/4000)
    print(nr_of_files)

    # load list of all files 
    chunk_files = sorted((embeddings_path).glob("chunk_*.parquet"))
    print(len(chunk_files))

    bert_embeddings = pd.read_parquet(
        chunk_files[:nr_of_files],
        columns=["id", "layer_last"],
    )

    if bert_embeddings.shape[0]> sample_size:
        bert_embeddings = bert_embeddings[:sample_size]

    bert_embeddings.shape

    ########### GARBAGE COLLECTION ############
    gc.collect()

    ########### CLUSTERING ###########
    X = numpy.stack(bert_embeddings.layer_last.to_numpy())  # shape (n_samples, 768)

    clusterer = DBSCAN(min_samples=minPts, eps=epsilon, metric='cosine')

    labels = clusterer.fit_predict(X)
    counter = collections.Counter(labels)

    return counter


In [26]:
# Apply DBSCAN and record the runtime
cluster_info = []
epsilon = 0.2
minPts = 380

for size in [500, 5000, 10000, 20000, 40000, 60000, 80000, 100000]:
    current_cluster = {}
    current_cluster["sample_size"] = size
    # set up time logging
    current_cluster["start_time"] = datetime.now().isoformat()
    start_ts = time.monotonic()  # for computing elapsed duration cheaply

    cluster_dist = get_clusters(size, epsilon=epsilon, minPts=minPts)

    current_cluster["end_time"] = datetime.now().isoformat()
    current_cluster["elapsed_seconds"] = round(time.monotonic() - start_ts, 1)
    current_cluster["epsilon"] = epsilon
    current_cluster["minPts"] = minPts
    current_cluster["cluster_dist"] = cluster_dist
    cluster_info.append(current_cluster)

1
124
2
124
3
124
5
124
10
124
15
124
20
124
25
124


In [27]:
print(cluster_info)

[{'sample_size': 500, 'start_time': '2026-08-19T17:50:27.843278', 'end_time': '2026-08-19T17:50:28.487844', 'elapsed_seconds': 0.6, 'epsilon': 0.2, 'minPts': 380, 'cluster_dist': Counter({np.int64(-1): 500})}, {'sample_size': 5000, 'start_time': '2026-08-19T17:50:28.487879', 'end_time': '2026-08-19T17:50:29.711013', 'elapsed_seconds': 1.2, 'epsilon': 0.2, 'minPts': 380, 'cluster_dist': Counter({np.int64(-1): 3889, np.int64(0): 1111})}, {'sample_size': 10000, 'start_time': '2026-08-19T17:50:29.711044', 'end_time': '2026-08-19T17:50:32.744828', 'elapsed_seconds': 3.0, 'epsilon': 0.2, 'minPts': 380, 'cluster_dist': Counter({np.int64(0): 5763, np.int64(-1): 4237})}, {'sample_size': 20000, 'start_time': '2026-08-19T17:50:32.744862', 'end_time': '2026-08-19T17:50:47.911815', 'elapsed_seconds': 15.2, 'epsilon': 0.2, 'minPts': 380, 'cluster_dist': Counter({np.int64(0): 14578, np.int64(-1): 5041, np.int64(1): 381})}, {'sample_size': 40000, 'start_time': '2026-08-19T17:50:47.911848', 'end_time':

In [19]:
cluster_info[0].keys()

dict_keys(['sample_size', 'start_time', 'end_time', 'elapsed_seconds', 'epsilon', 'minPts', 'cluster_dist'])

In [30]:
# Export the Results
with open("cluster_speed_kaggle_larger.jsonl", "w") as f:
    for item in cluster_info:
        item = item.copy()
        item["cluster_dist"] = {
            int(k): int(v)
            for k, v in item["cluster_dist"].items()
        }

        f.write(json.dumps(item) + "\n")